# CBN Sentinel: An AI powered AML Compliance Engine
**Automated Anti-Money Laundering (AML) & Suspicious Activity Reporting (SAR)**

## The Context
In 2024, the Central Bank of Nigeria mandated that all financial institutions that is banks, Fintechs e.t.c deployed real-time AML systems. With an 18-24 month compliance window, the industry is shifting from manual oversight to an AI-driven deterrence.

## "The Sentinel" 
This project is an end-to-end prototype of a modern compliance engine. It solves two critical problems:
* 1. **Detection:** The usage of Machine learning to identify suspicious transactions that human eyes could miss.
  2. **Reporting:** Using Generative AI to instantly draft a formal **Suspicious Activity Reports (SAR)** suitable for CBN submission.

## Tech Stack
* 1. Python & Scikit-Learn: For the implementation of **Isolation Forest** model for unsupervised anomaly detection.
  2. Large Language Models (LLMs): For the automation of the legal narrative for flagging transactions.
  3. Pandas: For the processing of high-volumne synthetic Nigerian fintech transactions.
  4. Streamlit: For the deployment of real-time monitoring dashboard for Compliance officers.

## Core Features
* 1. Anomaly Scoring: The end-product of this project can be able to detect round numbers traps, location spoofing, and sudden velocity spikes in funds transfer.
  2. One-Click SAR: The end product of this project can convert raw transaction metadata into a professional, legally-structured report.
  3. Compliance Heatmap: The end-product of this project can visualize high-risk zones for money laundering in real-time.

## Project's Impact
Through the automation of the detection-to-report pipeline, "The Sentinel" aims at reducing the time spent on the filing of a single suspicious case from **30 minutes to 5 minutes** and also ensuring that 100% compliance can be attained with zero human fatigue.

### Task One: Fintech Ledger Generator
For this project real bank data willn't be used and so a synthetic dataset is engineered. 

In [3]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [4]:
#For the generation of "Normal" transactions and "Suspicious" transactions
np.random.seed(42)
n_normal = 5000
n_anomalies = 50

#Section for the normal transaction
data_normal = {
    "Amount":np.random.gamma(shape=2, scale=10000, size=n_normal),
    "Transaction Type":np.random.choice(["Transfer to a Third-Party","Payment for Goods/Services","Withdrawal"], n_normal),
    "Sender's Location":np.random.choice(["Lagos","Abuja","Port Harcourt","Enugu","Ibadan"],n_normal),
    "Account Type":np.random.choice(["Corporate", "Individual"],n_normal),
    "Timestamp":np.sort(np.random.randint(0, 86400, size=n_normal)),
    "Is_anomaly":0
}
df_normal = pd.DataFrame(data_normal)

#Section for the large sum transaction that seems suspicious
individual_anomaly = n_anomalies // 2
corporate_anomaly = n_anomalies - individual_anomaly
amt_individual = np.random.uniform(5000001,10000000, size=individual_anomaly)
amt_corporate = np.random.uniform(10000001, 30000000, size=corporate_anomaly)
data_anomalies = {
    "Amount":np.concatenate([amt_individual, amt_corporate]),
    "Transaction Type":np.random.choice(["Transfer to a Third-Party","Payment for Goods/Services","Withdrawal"], n_anomalies),
    "Sender's Location":np.random.choice(["Lagos","Abuja","Port Harcourt","Enugu","Ibadan"],n_anomalies),
    "Account Type": ["Individual"] * individual_anomaly + ["Corporate"] * corporate_anomaly,
    "Timestamp":np.random.randint(0, 86400, size=n_anomalies),
    "Is_anomaly":1
}
df_anomalies = pd.DataFrame(data_anomalies)

# Generation of smurfers details
n_smurfer = 5
counts = np.random.randint(15,30, size=n_smurfer)
total_smurfers_rows = sum(counts)
smurfer_ids = np.repeat([f"Customer_Special {i:02d}" for i in range(1, n_smurfer + 1)],counts)
cities = ["Lagos","Abuja","Port Harcourt","Enugu","Ibadan"]
smurfer_locations = np.repeat(np.random.choice(cities,n_smurfer), counts)
smurf_amount = np.random.uniform(550000, 800000, size=total_smurfers_rows)

data_smurfing = {
    "Customer ID": smurfer_ids,
    "Amount":smurf_amount,
    "Transaction Type":np.random.choice(["Transfer to a Third-Party","Payment for Goods/Services","Withdrawal"], total_smurfers_rows),
    "Sender's Location":smurfer_locations,
    "Account Type": ["Individual"] * total_smurfers_rows,
    "Timestamp":np.random.randint(0,86400,total_smurfers_rows),
    "Is_anomaly":0
    
}
df_smurf = pd.DataFrame(data_smurfing)

#For ensuring that anomalsy cases and smurfing cases are evenly distributed within the dataset
df = pd.concat([df_normal, df_anomalies,df_smurf], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)
#For the Mapping of 1 and 0 in words to assume categorization
df["Remark"] = df["Is_anomaly"].map({0:"Normal", 1:"Suspicious"})

#For the generation of fictional customer id
df["Customer ID"] =df.apply(
    lambda row: row["Customer ID"] if "Special" in str(row["Customer ID"])
    else f"Customer {str(row.name + 1).zfill(2)}",
    axis=1
)
#df = df.drop(columns=["Is_Anomaly"])
df["Amount(\u20A6)"] = df["Amount"].round(2)

#Creation of the column on Account tiers in with the aim of providing more info
tiers = ["Tier 1", "Tier 2", "Tier 3"]
weight = [0.4,0.5,0.1]
df["Account Tier"] = np.random.choice(tiers, size=len(df),p=weight)
df.loc[df["Customer ID"].str.contains("Special"),"Account Tier"] = "Tier 3"
df["Account Tier"] = df["Account Tier"].astype("category")

#Conversion of time into readable format
def timeconvert(seconds):
    return str(timedelta(seconds=int(seconds)))

df["Timestamp"] = df["Timestamp"].apply(timeconvert)

#For the visualization of needed columns in the dataset
cols = ["Customer ID","Account Type","Account Tier","Amount(\u20A6)","Transaction Type", "Sender's Location", "Timestamp","Remark"]
df = df[cols]
#saving the data in csv format
df.to_csv("cbn_sentinel_data1.csv", index=False)

### Task Two: Data Importation, Feature Enginnering, Data Splitting
For the identification of velocity spikes, scaling round numbers, preprocessing categorical data
- The Amount column is enginnered for the flagging of Round number amount, flagging of ammounts above CBN's Five Million Threshold for Individuals and Ten Million Threshold for Corporations.
- Timestamp for the counting of transactions from a particular user in an hour
- Transaction Type for the understanding of different transactions and assigning of high risk scores to specfic locations

In [6]:
df = pd.read_csv("cbn_sentinel_data1.csv")
df.head() 

,Customer ID,Account Type,Account Tier,Amount(₦),Transaction Type,Sender's Location,Timestamp,Remark
0,Customer 01,Individual,Tier 2,16662.62,Withdrawal,Port Harcourt,8:30:11,Normal
1,Customer 02,Individual,Tier 2,23603.82,Transfer to a Third-Party,Abuja,5:07:44,Normal
2,Customer 03,Individual,Tier 2,32576.52,Payment for Goods/Services,Abuja,4:41:32,Normal
3,Customer 04,Corporate,Tier 1,2741.28,Payment for Goods/Services,Lagos,17:19:35,Normal
4,Customer 05,Corporate,Tier 1,15228.07,Payment for Goods/Services,Enugu,13:27:54,Normal


In [7]:
df["Txn_freq_24h"] = df.groupby("Customer ID")["Amount(\u20A6)"].transform("count")
df["Txn_volume"] = df.groupby("Customer ID")["Amount(\u20A6)"].transform("sum")
df["Avg_Txn_Size"] = df["Txn_volume"] / df["Txn_freq_24h"]
pd.options.display.float_format = "{:,.2f}".format #this tells pandas to show numbers in 2 decimal places instead of scientific notation
df["Round Number"] = df["Amount(\u20A6)"].apply(lambda x:1 if  x % 1000 == 0 else 0)

def behavioral_patrol(row):
    amt = row["Amount(\u20A6)"]
    acc_type = row["Account Type"]
    total_vol = row["Txn_volume"]
    freq = row["Txn_freq_24h"]

    limit = 5000000 if acc_type == "Individual" else 10000000
    above_threshold = 1 if amt > limit else 0
    is_structured = 1 if (total_vol > limit and amt < limit and freq > 1) else 0
    target = 1 if (above_threshold == 1 or is_structured == 1) else 0

    return pd.Series([above_threshold, is_structured, target])

df[["Above_threshold", "Is_structured", "Target"]] = df.apply(behavioral_patrol, axis=1)
df["Remark"] = df["Target"].map({0:"Normal", 1:"Suspicious"})

location_risk = {
    "Enugu": 4,  #High Cybercrime Zone
    "Ibadan": 4, #High Cybercrime Zone
    "Port Harcourt": 3, #High Cybercrime Zone
    "Abuja":3, #High Political Zone
    "Lagos":5 #Highest risk Zone
}
df["High_Risk_Location"] = df["Sender's Location"].map(location_risk).fillna(0)
df.head()

,Customer ID,Account Type,Account Tier,Amount(₦),Transaction Type,Sender's Location,Timestamp,Remark,Txn_freq_24h,Txn_volume,Avg_Txn_Size,Round Number,Above_threshold,Is_structured,Target,High_Risk_Location
0,Customer 01,Individual,Tier 2,"16,662.62",Withdrawal,Port Harcourt,8:30:11,Normal,1,"16,662.62","16,662.62",0,0,0,0,3
1,Customer 02,Individual,Tier 2,"23,603.82",Transfer to a Third-Party,Abuja,5:07:44,Normal,1,"23,603.82","23,603.82",0,0,0,0,3
2,Customer 03,Individual,Tier 2,"32,576.52",Payment for Goods/Services,Abuja,4:41:32,Normal,1,"32,576.52","32,576.52",0,0,0,0,3
3,Customer 04,Corporate,Tier 1,"2,741.28",Payment for Goods/Services,Lagos,17:19:35,Normal,1,"2,741.28","2,741.28",0,0,0,0,5
4,Customer 05,Corporate,Tier 1,"15,228.07",Payment for Goods/Services,Enugu,13:27:54,Normal,1,"15,228.07","15,228.07",0,0,0,0,4


In [8]:
#Preprocessing categorical and numerical column
ohe = OneHotEncoder(sparse_output=False,handle_unknown="ignore")
le = LabelEncoder()

categorical_cols = ["Transaction Type", "Account Type","Account Tier"]
encoded_features = ohe.fit_transform(df[categorical_cols])

#Creating the dataframe for storing the encoded features with proper names
encoded_df = pd.DataFrame(encoded_features, columns=ohe.get_feature_names_out(categorical_cols))

#Encoding the target variable "Remark"
df["Target"] = le.fit_transform(df["Remark"])

#combining the target columns to form the final training dataset
df_final = df.drop(columns=categorical_cols + ['Timestamp',"Sender's Location","Remark", "Customer ID"])
df_final = pd.concat([df_final,encoded_df], axis=1)

#For ensuring all column names are strings
df_final_columns = df_final.columns.astype(str)
df_final.columns.tolist()
df_final.head()
checkout = df_final[df_final["Is_structured"] == 1]
checkout.head()

,Amount(₦),Txn_freq_24h,Txn_volume,Avg_Txn_Size,Round Number,Above_threshold,Is_structured,Target,High_Risk_Location,Transaction Type_Payment for Goods/Services,Transaction Type_Transfer to a Third-Party,Transaction Type_Withdrawal,Account Type_Corporate,Account Type_Individual,Account Tier_Tier 1,Account Tier_Tier 2,Account Tier_Tier 3
18,"573,223.69",16,"10,909,943.79","681,871.49",0,0,1,1,4,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00
33,"679,140.20",20,"13,256,746.24","662,837.31",0,0,1,1,4,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00
101,"640,238.41",19,"12,615,535.99","663,975.58",0,0,1,1,3,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00
118,"774,950.68",22,"14,778,137.00","671,733.50",0,0,1,1,3,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00
119,"785,541.02",22,"14,778,137.00","671,733.50",0,0,1,1,3,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00


In [9]:
#Define Features (X) and Target(y) for model building
X = df_final.drop(columns= ["Target","Above_threshold","Is_structured"])
y = df_final["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples.")
X_train.columns

Training set: 4124 samples
Test set: 1032 samples.


Index(['Amount(₦)', 'Txn_freq_24h', 'Txn_volume', 'Avg_Txn_Size',
       'Round Number', 'High_Risk_Location',
       'Transaction Type_Payment for Goods/Services',
       'Transaction Type_Transfer to a Third-Party',
       'Transaction Type_Withdrawal', 'Account Type_Corporate',
       'Account Type_Individual', 'Account Tier_Tier 1', 'Account Tier_Tier 2',
       'Account Tier_Tier 3'],
      dtype='object')

### Model Selection and Training
For this project the best to use is the "Champion Challenger". Random Forest is selected as the baseline model for measuring the performance of even complex models. debugging features, checking model overkill not requiring heavier ml models, quantifyingg improvements
- Xgboost is the industry standard for stuctured tabular data. it builds trees squentially with each tree correcting the errors of the previous ones. this makes it good fore catching subtle fraud cases that basic models miss. it also handles imbalances in a dataset.
- Logistic Regression is for explanability. as it provides clear coefficients that is weights for every feature. it is also mathematically simple
- Isolation forest is for the unknown detection. it doesn't look at labelled columns to carry out any operation as it looks at data points that are different from everything else.

In [11]:
#1. Random Forest Classifier (Baseline)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

#2. XGBoost (The Challenger)
xgb_model = XGBClassifier(use_label_encoder=False,eval_metric="logloss")
xgb_model.fit(X_train,y_train)
xgb_preds = xgb_model.predict(X_test)

#3. Logistic Regression (The Explainable Model)
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

#4. Isolation Forest(The Anomaly Detector)
actual_ratio = 31/ 1032
iso_forest = IsolationForest(contamination=actual_ratio, random_state=42)
iso_preds = iso_forest.fit_predict(X_test)
iso_preds = [1 if x == -1 else 0 for x in iso_preds]

#Viewing comparison
model_results = {
    "Random Forest":rf_preds,
    "Logistic Regression": lr_preds,
    "XGBoost Model": xgb_preds,
    "Isolation Forest": iso_preds
}
for model_name, predictions in model_results.items():
    print(f"====== {model_name} =======")
    print(classification_report(y_test, predictions))
    print("\n")

print(f"Random Forest: {accuracy_score(y_test, rf_preds):.2%}")
print(f"XGBoost Model: {accuracy_score(y_test, xgb_preds):.2%}")
print(f"Logistic Regression: {accuracy_score(y_test, lr_preds):.2%}")
print(f"Isolation Forest: {accuracy_score(y_test, iso_preds):.2%}")

====== Random Forest =======
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1001
           1       1.00      1.00      1.00        31

    accuracy                           1.00      1032
   macro avg       1.00      1.00      1.00      1032
weighted avg       1.00      1.00      1.00      1032



====== Logistic Regression =======
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1001
           1       1.00      1.00      1.00        31

    accuracy                           1.00      1032
   macro avg       1.00      1.00      1.00      1032
weighted avg       1.00      1.00      1.00      1032



====== XGBoost Model =======
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1001
           1       1.00      1.00      1.00        31

    accuracy                           1.00      1032
   macro avg       1.00      1.

In [12]:
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({"Feature":feature_names, "Importance":importances})
print(feature_importance_df.sort_values(by="Importance", ascending=False))

                                        Feature  Importance
0                                     Amount(₦)        0.33
2                                    Txn_volume        0.29
3                                  Avg_Txn_Size        0.23
1                                  Txn_freq_24h        0.12
13                          Account Tier_Tier 3        0.02
11                          Account Tier_Tier 1        0.00
12                          Account Tier_Tier 2        0.00
9                        Account Type_Corporate        0.00
10                      Account Type_Individual        0.00
5                            High_Risk_Location        0.00
8                   Transaction Type_Withdrawal        0.00
7    Transaction Type_Transfer to a Third-Party        0.00
6   Transaction Type_Payment for Goods/Services        0.00
4                                  Round Number        0.00


In [13]:
from sklearn.preprocessing import StandardScaler
scaling = StandardScaler()
iso_forest.fit(scaling.fit_transform(X_train))

IsolationForest(contamination=0.030038759689922482, random_state=42)

In [14]:
X_test_scaled = scaling.transform(X_test)
iso_raw_preds = iso_forest.predict(X_test_scaled)
iso_final_preds = np.where(iso_raw_preds == -1,1,0)
print(f"------- Isolation Forest Results -----------")
print(classification_report(y_test, iso_final_preds))

------- Isolation Forest Results -----------
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1001
           1       1.00      0.97      0.98        31

    accuracy                           1.00      1032
   macro avg       1.00      0.98      0.99      1032
weighted avg       1.00      1.00      1.00      1032



In [15]:
scores = iso_forest.decision_function(X_test_scaled)
#Adding to a temporary database
results_df = X_test.copy()
results_df["Anomaly_Score"] = scores
results_df["Actual_Label"] = y_test

#Looking at Actual Score Vs Fraud
print(results_df.groupby("Actual_Label")["Anomaly_Score"].mean())

Actual_Label
0    0.16
1   -0.04
Name: Anomaly_Score, dtype: float64


In [16]:
detected_anomalies = results_df[results_df["Actual_Label"]==1]
detected_anomalies.head()

,Amount(₦),Txn_freq_24h,Txn_volume,Avg_Txn_Size,Round Number,High_Risk_Location,Transaction Type_Payment for Goods/Services,Transaction Type_Transfer to a Third-Party,Transaction Type_Withdrawal,Account Type_Corporate,Account Type_Individual,Account Tier_Tier 1,Account Tier_Tier 2,Account Tier_Tier 3,Anomaly_Score,Actual_Label
468,"18,281,919.24",1,"18,281,919.24","18,281,919.24",0,3,0.00,0.00,1.00,1.00,0.00,0.00,1.00,0.00,-0.05,1
3955,"5,491,326.93",1,"5,491,326.93","5,491,326.93",0,5,0.00,0.00,1.00,0.00,1.00,1.00,0.00,0.00,0.00,1
33,"679,140.20",20,"13,256,746.24","662,837.31",0,4,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,-0.03,1
5146,"684,365.59",22,"14,778,137.00","671,733.50",0,3,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,-0.06,1
3299,"6,534,679.41",1,"6,534,679.41","6,534,679.41",0,3,0.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00,-0.01,1


In [17]:
import joblib
joblib.dump(iso_forest,"AntiMoneyLaundering_Model.pkl")
joblib.dump(scaling,"AML_scaler.pkl")
joblib.dump(ohe, "AML_encoder.pkl")
print("Model and Scaler and encoder Successfuly exported")

Model and Scaler and encoder Successfuly exported


In [18]:
#For visualizing the features that influence the model's decision
from sklearn.metrics import f1_score
from sklearn.inspection import permutation_importance
#f1_scorer = make_scorer(f1_score,average="macro")
def iso_forest_scorer(estimator, X,y):
    raw_preds = estimator.predict(X)
    translated_preds = np.where(raw_preds == -1,1,0)
    return f1_score(y, translated_preds, average="macro")
results = permutation_importance(
    iso_forest,
    X_test_scaled,
    y_test,
    scoring=iso_forest_scorer,
    n_repeats=10,
    random_state=42
)
importance_df = pd.DataFrame({"Feature":X_test.columns, 
                              "Importance":results.importances_mean})
print(importance_df.sort_values(by="Importance",ascending=False))

                                        Feature  Importance
2                                    Txn_volume        0.16
1                                  Txn_freq_24h        0.10
3                                  Avg_Txn_Size        0.09
0                                     Amount(₦)        0.09
13                          Account Tier_Tier 3        0.04
9                        Account Type_Corporate        0.01
7    Transaction Type_Transfer to a Third-Party        0.01
4                                  Round Number        0.00
5                            High_Risk_Location        0.00
8                   Transaction Type_Withdrawal        0.00
10                      Account Type_Individual        0.00
11                          Account Tier_Tier 1       -0.00
12                          Account Tier_Tier 2       -0.00
6   Transaction Type_Payment for Goods/Services       -0.01


### Task Three: AI Model Integration
In line with CBN's Baseline Standards for Automated AML Solutions which specifically mandates that financial institutions automate reporting to the Nigerian Financial Intelligence Unit (NFIU) within strict timelines. Writing a Supicious Activity Report using Groq LLM takes a compliance officer up to 45 minutes, this can be reeduced to five minutes

In [20]:
results_df = df.loc[X_test.index].copy()
results_df["AI_Prediction"] = np.where(iso_forest.predict(X_test_scaled) == -1,1,0)
results_df["Anomaly_Score"] = iso_forest.decision_function(X_test_scaled)
cases = results_df[results_df["AI_Prediction"] == 1]
print(cases[["Customer ID","AI_Prediction"]].head(1))

      Customer ID  AI_Prediction
468  Customer 469              1


In [21]:
def sar_generator_prompt(row):
    if row["Txn_freq_24h"] > 5:
        red_flag = "Structuring/Smurfing(High velocity of small transactions)"
    else:
        red_flag = "Large Value Transaction (Single breach of CBN Transaction Limit)"
    return f"""
    ----Case File: {row["Customer ID"]} ----
    Account Tier: {row["Account Tier"]}
    Location: {row["Sender's Location"]}
    Total Volume: \u20A6 {row["Txn_volume"]:,.2f}
    Frequency: {row["Txn_freq_24h"]} transaction(s) in 24 hours
    Average Transaction: \u20A6 {row["Avg_Txn_Size"]:,.2f}
    Behavioral Red Flags: {red_flag}
    """

case_evidence = sar_generator_prompt(cases.iloc[1])
print(case_evidence)


    ----Case File: Customer_Special 04 ----
    Account Tier: Tier 3
    Location: Ibadan
    Total Volume: ₦ 13,256,746.24
    Frequency: 20 transaction(s) in 24 hours
    Average Transaction: ₦ 662,837.31
    Behavioral Red Flags: Structuring/Smurfing(High velocity of small transactions)
    


In [54]:
import os
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from datetime import datetime

In [27]:
#Setting up environment
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("Groq API Key not found. AML Sentinel offline😏")
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.1, groq_api_key=api_key)
print("Key safely loaded from .env. AML Sentinel online😊")
template_text = """
    ROLE: Senior AML Compliance Officer, Central Bank of Nigeria (CBN).
    TASK: Draft a formal Suspicious Activity Report (SAR) for the NFIU.
    
    DATA EVIDENCE:
    - Date: (current_date)
    - Subject: {subject}
    - Account Tier: {account_tier}
    - Total Volume: NGN {total_volume}
    - Transaction Count: {transaction_freq} in 24 hours
    - Avg Transaction: NGN {avg_transaction}
    - Location: {location}
    - Violation: {violation}
    
    LEGAL REQUIREMENT: 
    Reference the 'Money Laundering (Prevention and Prohibition) Act 2022'. 
    Explain why this behavior constitutes {violation}.
    Recommend a 'Post-No-Debit' (PND) status.
    """
sar_prompt = PromptTemplate.from_template(template_text)
sar_chain = sar_prompt | llm

def draft_sar_report(row):
    """Function to call Groq AI for a professional CBN SAR draft"""
    # Context for the AI based on our Behavioral Patrol
    current_date =datetime.now().strftime("%B %d, %Y")
    txn_freq =row.get("Txn_freq_24h", 0)
    violation = "Structuring/Smurfing" if txn_freq > 10 else "Threshold Breach"
    formatted_prompt = sar_prompt.format( 
    subject=row.get('Customer ID'),
    account_tier=row.get('Account Tier'),
    total_volume=f"{row.get('Txn_volume',0):,.2f}",
    transaction_freq=txn_freq,
    avg_transaction=f"{row.get('Avg_Txn_Size',0):,.2f}",
    location=row.get("Sender's Location"),
    violation=violation        
    )
    response = llm.invoke(formatted_prompt)
    return response.content

# RUN THE LOOP & SAVE
print(f"Starting AI drafting for {len(cases)} cases... Please wait.")
cases['AI_SAR_Draft'] = cases.apply(draft_sar_report, axis=1)

# EXPORT TO CSV (For GitHub/LinkedIn visibility)
cases.to_csv("CBN_Sentinel_Final_Reports.csv", index=False)
print("Success! Reports saved to 'CBN_Sentinel_Final_Reports.csv'.")

# View the first one
print("\n--- SAMPLE REPORT PREVIEW ---")
print(cases['AI_SAR_Draft'].iloc[0])
#Control + Shift + L is a multiselect way for changing a variable name on multiple front

Key safely loaded from .env. AML Sentinel online😊
Starting AI drafting for 30 cases... Please wait.
Success! Reports saved to 'CBN_Sentinel_Final_Reports.csv'.

--- SAMPLE REPORT PREVIEW ---
**SUSPICIOUS ACTIVITY REPORT (SAR)**

**Date:** March 27, 2026

**Reporting Institution:** Central Bank of Nigeria (CBN)
**Reporter:** [Your Name], Senior AML Compliance Officer

**Subject:** Customer 469
**Account Tier:** Tier 2
**Location:** Port Harcourt

**Summary of Suspicious Activity:**
On March 27, 2026, our monitoring systems detected a suspicious transaction involving Customer 469, a Tier 2 account holder. The transaction details are as follows:

* **Total Volume:** NGN 18,281,919.24
* **Transaction Count:** 1 transaction within a 24-hour period
* **Average Transaction Value:** NGN 18,281,919.24

**Reason for Suspicion:**
The transaction exceeds the threshold limits set by the Central Bank of Nigeria, constituting a Threshold Breach. According to Section 10 of the Money Laundering (Preven

In [28]:
#Setting up environment
load_dotenv()
api_key= os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("Groq API Key not found. AML Sentinel offline😏")
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.4, groq_api_key=api_key)
print("Key safely loaded from .env. AML Sentinel online😊")

template_text = """
    ROLE: Senior AML Compliance Officer, Central Bank of Nigeria (CBN).
    TASK: Draft a formal Suspicious Activity Report (SAR) for the NFIU.
    
    DATA EVIDENCE:
    - Date: (current_date)
    - Subject: {subject}
    - Account Tier: {account_tier}
    - Total Volume: NGN {total_volume}
    - Transaction Count: {transaction_freq} in 24 hours
    - Avg Transaction: NGN {avg_transaction}
    - Location: {location}
    - Violation: {violation}
    
    LEGAL REQUIREMENT: 
    Reference the 'Money Laundering (Prevention and Prohibition) Act 2022'. 
    Explain why this behavior constitutes {violation}.
    Recommend a 'Post-No-Debit' (PND) status.
    """
# Integrating the llm with the prompt
sar_prompt = PromptTemplate.from_template(template_text)
sar_chain = sar_prompt | llm

def draft_sar_report(row):
    """Function to call Groq AI for a professional CBN SAR draft"""
    # Context for the AI based on our Behavioral Patrol
    current_date =datetime.now().strftime("%B %d, %Y")
    txn_freq =row.get("Txn_freq_24h", 0)
    violation = "Structuring/Smurfing" if txn_freq > 10 else "Threshold Breach"
    response = sar_chain.invoke({ 
        "subject":row.get('Customer ID'),
        "account_tier":row.get('Account Tier'),
        "total_volume":f"{row.get('Txn_volume',0):,.2f}",
        "transaction_freq":txn_freq,
        "avg_transaction":f"{row.get('Avg_Txn_Size',0):,.2f}",
        "location":row.get("Sender's Location"),
        "violation":violation        
    })
    return response.content

# RUN THE LOOP & SAVE
print(f"Starting AI drafting for {len(cases)} cases... Please wait.")
cases['AI_SAR_Draft'] = cases.apply(draft_sar_report, axis=1)

# EXPORT TO CSV (For GitHub/LinkedIn visibility)
cases.to_csv("CBN_Sentinel_Final_Reports.csv", index=False)
print("Success! Reports saved to 'CBN_Sentinel_Final_Reports.csv'.")

# View the first one
print("\n--- SAMPLE REPORT PREVIEW ---")
print(cases['AI_SAR_Draft'].iloc[1])

Key safely loaded from .env. AML Sentinel online😊
Starting AI drafting for 30 cases... Please wait.
Success! Reports saved to 'CBN_Sentinel_Final_Reports.csv'.

--- SAMPLE REPORT PREVIEW ---
**SUSPICIOUS ACTIVITY REPORT (SAR)**

**Date:** March 27, 2026

**Reference Number:** CBN/SAR/2026/001

**Subject:** Customer_Special 04

**Introduction:**
Pursuant to Section 6 of the Money Laundering (Prevention and Prohibition) Act 2022, which mandates the reporting of suspicious transactions to the Nigerian Financial Intelligence Unit (NFIU), this report is submitted to bring to your attention a suspicious activity involving Customer_Special 04, a Tier 3 account holder.

**Summary of Suspicious Activity:**
On March 27, 2026, our monitoring systems detected unusual transaction patterns on the account of Customer_Special 04. The key highlights of the suspicious activity are as follows:

- **Total Volume:** NGN 13,256,746.24
- **Transaction Count:** 20 transactions within a 24-hour period
- **Aver

In [52]:
%%writefile sentinel_app.py
import streamlit as st
import pandas as pd
import os
import numpy as np
import plotly.express as px
import joblib
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from datetime import datetime
from fpdf import FPDF

# Setting Up Assets
st.set_page_config(page_title="The Sentinel: AML Compliance Engine", layout="wide")
load_dotenv()
#Secure API Key for Deployment 
api_key = st.secrets.get("GROQ_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("Groq API Key not found. AML Sentinel offline😏")

@st.cache_resource
def load_ml_assets():
    model = joblib.load("AntiMoneyLaundering_Model.pkl")
    scaler = joblib.load("AML_scaler.pkl")
    encoder = joblib.load("aml_encoder.pkl")
    return model, scaler, encoder

model, scaler, encoder = load_ml_assets()
#For the conversion of .txt file into pdf
def generate_sar_pdf(text):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    clean_text = text.replace("₦","NGN").replace("\u20A6","NGN")
    pdf.multi_cell(0, 10, txt=clean_text)
    return pdf.output(dest="S").encode('latin-1')
# LLM SAR Prompt Drafting
llm = ChatGroq(temperature = 0.1,model_name="llama-3.3-70b-versatile",groq_api_key = api_key)
template_text = """
    ROLE: Senior AML Compliance Officer, Central Bank of Nigeria (CBN).
    TASK: Draft a formal Suspicious Activity Report (SAR) for the NFIU.
    
    DATA EVIDENCE:
    - Date: (current_date)
    - Subject: {subject}
    - Account Tier: {account_tier}
    - Total Volume: NGN {total_volume}
    - Transaction Count: {transaction_freq} in 24 hours
    - Avg Transaction: NGN {avg_transaction}
    - Location: {location}
    - Violation: {violation}
    
    LEGAL REQUIREMENT: 
    Reference the 'Money Laundering (Prevention and Prohibition) Act 2022'. 
    Explain why this behavior constitutes {violation}.
    Recommend a 'Post-No-Debit' (PND) status.
    """
# Integrating the llm with the prompt
sar_prompt = PromptTemplate.from_template(template_text)
sar_chain = sar_prompt | llm

#Sidebar and Frontpage Customization
st.title("🛡️ AML Sentinel Compliance Engine")
st.markdown("An effort at enforcing compliance to CBN policy")
st.sidebar.title("Data Holder")
st.sidebar.caption("Upload your dataset here")
st.sidebar.info("files received are csv or excel and not both")
uploaded_file = st.sidebar.file_uploader("Dataset uploading", type=["csv","xlsx"])
#Default Dataset provided 
default_dataset = "cbn_sentinel_data1.csv"
if uploaded_file is not None:
    if uploaded_file.name.endswith(".csv"):
        df = pd.read_csv(uploaded_file)
    else:
        df = pd.read_excel(uploaded_file)
    st.sidebar.success("Data successfully uploaded")
elif os.path.exists(default_dataset):
    df = pd.read_csv(default_dataset)
    st.sidebar.info("Using default dataset")
else:
    st.info("No file found")
    
# Feature Engineerin g for Higher level model's performance
if df is not None:
    amount_column = "Amount(\u20A6)"
    df["Txn_freq_24h"] = df.groupby("Customer ID")[amount_column].transform("count")
    df["Txn_volume"] = df.groupby("Customer ID")[amount_column].transform("sum")
    df["Avg_Txn_Size"] = df["Txn_volume"] / df["Txn_freq_24h"]
    pd.options.display.float_format = "{:,.2f}".format #this tells pandas to show numbers in 2 decimal places instead of scientific notation
    df["Round Number"] = df[amount_column].apply(lambda x:1 if  x % 1000 == 0 else 0)
    location_risk = {
        "Enugu": 4,  #High Cybercrime Zone
        "Ibadan": 4, #High Cybercrime Zone
        "Port Harcourt": 3, #High Cybercrime Zone
        "Abuja":3, #High Political Zone
        "Lagos":5 #Highest risk Zone
    }
    df["High_Risk_Location"] = df["Sender's Location"].map(location_risk).fillna(0)

numeric_features = [amount_column,"Txn_freq_24h","Txn_volume","Avg_Txn_Size","Round Number","High_Risk_Location"]
categorical_features = ["Transaction Type","Account Type","Account Tier"]
encoded_categories = encoder.transform(df[categorical_features])
if hasattr(encoded_categories,"toarray"):
    encoded_categories = encoded_categories.toarray()

X_combine = np.hstack([df[numeric_features].values, encoded_categories])
scaled_data = scaler.transform(X_combine)
modeler = model.predict(scaled_data)
df["Risk_Status"] =["Flagged" if x == -1 else "Clear" for x in modeler]

#Dashboard Visual
st.write("### Data Preview")
st.dataframe(df.head(), use_container_width=True)
st.subheader("📊Sentinel Intelligence Map")
fig = px.scatter(
df,
x="Txn_volume",#for showing structuring
y="Txn_freq_24h",#For showing velocity
color="Risk_Status",
color_discrete_map={"Flagged": "#EF553B","Clear":"#00CC96"},
hover_data=[amount_column,"Account Tier","Transaction Type"],
title="Sentinel Intelligence: Transaction Risk Map",
template="plotly_dark"
)
st.plotly_chart(fig,use_container_width=True)
flag_df = df[df["Risk_Status"] == "Flagged"]

if not flag_df.empty:
    st.subheader("Flagged Transactions")
    st.dataframe(flag_df, use_container_width=True)
    st.divider()
    st.subheader("Automated SAR Narrative")
    selected_idx = st.selectbox("Select Transaction Index to report from the dropdown menu", flag_df.index)
    record = flag_df.loc[selected_idx]
    if "sar_report" not in st.session_state:
        st.session_state.sar_report = None
    if "last_index" not in st.session_state or st.session_state.last_index != selected_idx:
        st.session_state.sar_report = None
        st.session_state.last_index = selected_idx
    if st.button("Generate SAR Draft"):
        with st.spinner("Drafting CBN_Compliant Report..."):
            current_date = datetime.now().strftime("%B %d, %Y"),
            txn_freq =record.get("Txn_freq_24h", 0)
            violation = "Structuring/Smurfing" if txn_freq > 10 else "Threshold Breach"
            record_input = { 
                "subject":record.get('Customer ID'),
                "account_tier":record.get('Account Tier'),
                "total_volume":f"{record.get('Txn_volume',0):,.2f}",
                "transaction_freq":txn_freq,
                "avg_transaction":f"{record.get('Avg_Txn_Size',0):,.2f}",
                "location":record.get("Sender's Location"),
                "violation":violation
            }
            response = sar_chain.invoke(record_input)
            st.session_state.sar_report=response.content

    if st.session_state.sar_report:
        st.info(st.session_state.sar_report)
        pdf_bytes = generate_sar_pdf(st.session_state.sar_report)
        st.download_button(
            label="Download Official SAR (.PDF)",
            data=pdf_bytes,
            file_name=f"SAR_{selected_idx}.pdf",
            mime="application/pdf"
        )
        st.download_button(
            label="Download Official SAR (.txt)",
            data=st.session_state.sar_report,
            file_name=f"SAR_{selected_idx}.txt"
        )
    
else:
    st.success("Clean Sweep: No anomalies detected")

Overwriting sentinel_app.py


In [48]:
from fpdf import FPDF
        pdf_bytes = generate()
        pdf_bytes = generate()
st.download_button("Download Official SAR (.txt)",
                            st.session_state.sar_report,
                           file_name=f"SAR_{selected_idx}.txt")